In [18]:
# Cell 0: Fix Path Issues - Run This First
import os
import sys
from pathlib import Path

# Force the correct project root
PROJECT_ROOT = Path(r"C:\Users\nyvra\Downloads\sp500-predictor")
os.chdir(PROJECT_ROOT)

# Verify we're in the right place
print(f"✅ Working directory: {Path.cwd()}")
print(f"✅ Data exists: {(PROJECT_ROOT / 'data' / 'processed' / 'sp500_processed.csv').exists()}")
print(f"✅ Models exist: {(PROJECT_ROOT / 'models' / 'ensembles' / 'final_model.pkl').exists()}")

# Add to path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Project root set to: {PROJECT_ROOT}")

✅ Working directory: C:\Users\nyvra\Downloads\sp500-predictor
✅ Data exists: True
✅ Models exist: True
✅ Project root set to: C:\Users\nyvra\Downloads\sp500-predictor


In [26]:
# ============================================================
# COMPLETE S&P 500 PREDICTOR PIPELINE - WORKING VERSION
# ============================================================

# Cell 1: Imports and Setup
import sys
import os
import json
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# ML imports
import yfinance as yf
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

# Set paths - FORCED to correct location
PROJECT_ROOT = Path(r"C:\Users\nyvra\Downloads\sp500-predictor")
DATA_PATH = PROJECT_ROOT / 'data'
MODELS_PATH = PROJECT_ROOT / 'models'
LOGS_PATH = PROJECT_ROOT / 'logs'
PROCESSED_DATA_PATH = DATA_PATH / 'processed' / 'sp500_processed.csv'

print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📁 Data path: {DATA_PATH}")
print(f"📁 Models path: {MODELS_PATH}")
print(f"📁 Processed data exists: {PROCESSED_DATA_PATH.exists()}")


# Cell 2: Data Collector
class DataCollector:
    def fetch_data(self, start_date='2010-01-01', end_date=None):
        if end_date is None:
            end_date = datetime.now().strftime('%Y-%m-%d')
        print(f"📊 Fetching data from {start_date} to {end_date}")
        ticker = yf.Ticker("^GSPC")
        df = ticker.history(start=start_date, end=end_date)
        if df.empty:
            raise ValueError("No data fetched")
        df.columns = [col.lower() for col in df.columns]
        df['returns'] = df['close'].pct_change()
        df['target'] = df['close'].shift(-5) / df['close'] - 1
        df = df.dropna()
        print(f"✅ Fetched {len(df)} rows")
        return df
    
    def create_features(self, df):
        df['sma_20'] = df['close'].rolling(20).mean()
        df['sma_50'] = df['close'].rolling(50).mean()
        df['volatility'] = df['returns'].rolling(20).std() * np.sqrt(252)
        df['close_vs_sma20'] = (df['close'] - df['sma_20']) / df['sma_20']
        df['close_vs_sma50'] = (df['close'] - df['sma_50']) / df['sma_50']
        df['volume_ratio'] = df['volume'] / df['volume'].rolling(20).mean()
        df = df.dropna()
        feature_cols = ['open', 'high', 'low', 'close', 'volume', 'returns',
                       'volatility', 'close_vs_sma20', 'close_vs_sma50', 'volume_ratio']
        X = df[feature_cols].values
        y = df['target'].values
        return X, y, feature_cols
    
    def get_train_test_split(self, X, y, test_size=0.2):
        split_idx = int(len(X) * (1 - test_size))
        return X[:split_idx], X[split_idx:], y[:split_idx], y[split_idx:]

print("✅ DataCollector ready")


# Cell 3: Training Pipeline (UPDATED - Handles both model formats)
class TrainingPipeline:
    def __init__(self):
        self.data_collector = DataCollector()
        self.imputer = SimpleImputer(strategy='mean')
        self.scaler = StandardScaler()
        self.model = None
        
    def train(self, retrain_reason='manual'):
        print("\n" + "="*60)
        print("🚀 STARTING TRAINING")
        print("="*60)
        
        df = self.data_collector.fetch_data()
        X, y, features = self.data_collector.create_features(df)
        
        X_train, X_test, y_train, y_test = self.data_collector.get_train_test_split(X, y)
        
        X_train = self.imputer.fit_transform(X_train)
        X_test = self.imputer.transform(X_test)
        X_train = self.scaler.fit_transform(X_train)
        X_test = self.scaler.transform(X_test)
        
        self.model = CatBoostRegressor(iterations=500, depth=6, learning_rate=0.1, 
                                        random_seed=42, verbose=False)
        self.model.fit(X_train, y_train)
        
        y_pred = self.model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        direction_acc = (np.sign(y_test) == np.sign(y_pred)).mean()
        
        artifacts = {
            'model': self.model,
            'imputer': self.imputer,
            'scaler': self.scaler,
            'feature_names': features,
            'metrics': {'test_rmse': float(rmse), 'direction_accuracy': float(direction_acc)},
            'training_date': datetime.now().isoformat()
        }
        
        joblib.dump(artifacts, MODELS_PATH / 'production_model.pkl')
        print(f"✅ Model trained! RMSE: {rmse:.4f}")
        return artifacts
    
    def load_model(self):
        """Load production model - handles both old and new formats"""
        model_path = MODELS_PATH / 'production_model.pkl'
        if not model_path.exists():
            raise FileNotFoundError(f"No production model found at {model_path}")
        
        artifacts = joblib.load(model_path)
        
        # Handle different model formats
        if 'model' in artifacts:
            self.model = artifacts['model']
        else:
            self.model = artifacts
        
        # Handle imputer and scaler
        if 'imputer' in artifacts:
            self.imputer = artifacts['imputer']
        else:
            self.imputer = SimpleImputer(strategy='mean')
            
        if 'scaler' in artifacts:
            self.scaler = artifacts['scaler']
        else:
            self.scaler = StandardScaler()
        
        # Handle feature names
        if 'feature_names' in artifacts:
            self.feature_names = artifacts['feature_names']
        
        # Handle metrics safely
        metrics = artifacts.get('metrics', {})
        rmse = metrics.get('test_rmse', 0.0)
        
        print(f"✅ Model loaded (RMSE: {rmse:.4f})")
        return artifacts

print("✅ TrainingPipeline ready")

# Cell 4: Prediction Pipeline (No API - Uses Local Data)
class PredictionPipeline:
    def __init__(self):
        self.trainer = TrainingPipeline()
        
    def predict_from_local(self):
        """Predict using last available data - NO API CALLS"""
        print("\n🔮 Making prediction from local data...")
        
        artifacts = self.trainer.load_model()
        
        # Load processed data
        df = pd.read_csv(PROCESSED_DATA_PATH, index_col=0, parse_dates=True)
        
        # Calculate features for the last row
        df['returns'] = df['close'].pct_change()
        df['sma_20'] = df['close'].rolling(20).mean()
        df['sma_50'] = df['close'].rolling(50).mean()
        df['volatility'] = df['returns'].rolling(20).std() * np.sqrt(252)
        df['close_vs_sma20'] = (df['close'] - df['sma_20']) / df['sma_20']
        df['close_vs_sma50'] = (df['close'] - df['sma_50']) / df['sma_50']
        df['volume_ratio'] = df['volume'] / df['volume'].rolling(20).mean()
        
        df = df.dropna()
        last_row = df.iloc[-1]
        last_date = df.index[-1]
        current_price = last_row['close']
        
        features = np.array([[
            last_row['open'], last_row['high'], last_row['low'], last_row['close'],
            last_row['volume'], last_row['returns'], last_row['volatility'],
            last_row['close_vs_sma20'], last_row['close_vs_sma50'], last_row['volume_ratio']
        ]])
        
        features = artifacts['imputer'].transform(features)
        features = artifacts['scaler'].transform(features)
        prediction = artifacts['model'].predict(features)[0]
        
        # Determine confidence and recommendation
        if prediction > 0.02:
            confidence = "High"
            recommendation = "Strong BUY"
        elif prediction > 0:
            confidence = "Medium"
            recommendation = "Cautious BUY"
        elif prediction > -0.02:
            confidence = "Low"
            recommendation = "HOLD"
        else:
            confidence = "Medium"
            recommendation = "SELL"
        
        result = {
            'prediction': float(prediction),
            'prediction_percent': f"{prediction:.4%}",
            'direction': 'BULLISH' if prediction > 0 else 'BEARISH',
            'confidence': confidence,
            'recommendation': recommendation,
            'current_price': float(current_price),
            'date': last_date.strftime('%Y-%m-%d'),
            'timestamp': datetime.now().isoformat()
        }
        
        print(f"\n📊 Market Data (as of {result['date']}):")
        print(f"   Current S&P 500: ${result['current_price']:,.2f}")
        print(f"\n🎯 PREDICTION:")
        print(f"   Expected Next Week Return: {result['prediction_percent']}")
        print(f"   Direction: {result['direction']}")
        print(f"   Confidence: {result['confidence']}")
        print(f"   Recommendation: {result['recommendation']}")
        
        return result

print("✅ PredictionPipeline ready")


# Cell 5: Pipeline Orchestrator
class PipelineOrchestrator:
    def __init__(self):
        self.trainer = TrainingPipeline()
        self.predictor = PredictionPipeline()
    
    def run_full_training(self, retrain_reason='manual'):
        return self.trainer.train(retrain_reason=retrain_reason)
    
    def run_prediction(self):
        return self.predictor.predict_from_local()

print("✅ PipelineOrchestrator ready")


# Cell 6: Local Training Function (No API Calls)
def train_from_local_data():
    """Train model using existing processed data only - NO API CALLS"""
    
    print("\n" + "="*60)
    print("🚀 TRAINING FROM LOCAL CACHE (No API Calls)")
    print("="*60)
    
    # Load processed data
    processed_path = PROCESSED_DATA_PATH
    
    if not processed_path.exists():
        raise FileNotFoundError(f"No processed data found at {processed_path}")
    
    df = pd.read_csv(processed_path, index_col=0, parse_dates=True)
    print(f"✅ Loaded {len(df)} rows from local cache")
    
    # Create features
    df['returns'] = df['close'].pct_change()
    df['sma_20'] = df['close'].rolling(20).mean()
    df['sma_50'] = df['close'].rolling(50).mean()
    df['volatility'] = df['returns'].rolling(20).std() * np.sqrt(252)
    df['close_vs_sma20'] = (df['close'] - df['sma_20']) / df['sma_20']
    df['close_vs_sma50'] = (df['close'] - df['sma_50']) / df['sma_50']
    df['volume_ratio'] = df['volume'] / df['volume'].rolling(20).mean()
    df['target'] = df['close'].shift(-5) / df['close'] - 1
    df = df.dropna()
    
    print(f"✅ After feature engineering: {len(df)} rows")
    
    # Feature columns
    feature_cols = ['open', 'high', 'low', 'close', 'volume', 'returns',
                    'volatility', 'close_vs_sma20', 'close_vs_sma50', 'volume_ratio']
    
    X = df[feature_cols].values
    y = df['target'].values
    
    print(f"✅ Features: {len(feature_cols)}")
    print(f"✅ Samples: {len(y)}")
    
    # Train/test split
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    
    print(f"📊 Train: {len(X_train)} samples, Test: {len(X_test)} samples")
    
    # Preprocess
    imputer = SimpleImputer(strategy='mean')
    scaler = StandardScaler()
    
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # Train model
    print("\n🤖 Training CatBoost model...")
    model = CatBoostRegressor(
        iterations=500,
        depth=6,
        learning_rate=0.1,
        random_seed=42,
        verbose=False
    )
    model.fit(X_train, y_train)
    
    # Evaluate
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    direction_acc = (np.sign(y_test) == np.sign(y_pred)).mean()
    
    print(f"\n📈 Performance:")
    print(f"   Test RMSE: {rmse:.4f} ({rmse*100:.2f}%)")
    print(f"   Direction Accuracy: {direction_acc:.2%}")
    
    # Save model
    artifacts = {
        'model': model,
        'imputer': imputer,
        'scaler': scaler,
        'feature_names': feature_cols,
        'metrics': {
            'test_rmse': float(rmse),
            'direction_accuracy': float(direction_acc)
        },
        'training_date': datetime.now().isoformat(),
        'retrain_reason': 'local_cache'
    }
    
    joblib.dump(artifacts, MODELS_PATH / 'production_model.pkl')
    print(f"\n✅ Model saved to {MODELS_PATH / 'production_model.pkl'}")
    
    return artifacts


# Cell 7: Local Prediction Function
def predict_from_local():
    """Make prediction using last available data point - NO API CALLS"""
    
    print("\n" + "="*60)
    print("🔮 MAKING PREDICTION FROM LOCAL DATA")
    print("="*60)
    
    # Load model
    model_path = MODELS_PATH / 'production_model.pkl'
    if not model_path.exists():
        print("❌ No model found. Run training first.")
        return None
    
    artifacts = joblib.load(model_path)
    model = artifacts['model']
    imputer = artifacts['imputer']
    scaler = artifacts['scaler']
    
    print(f"✅ Model loaded (RMSE: {artifacts['metrics']['test_rmse']:.4f})")
    
    # Load the last data point from processed data
    df = pd.read_csv(PROCESSED_DATA_PATH, index_col=0, parse_dates=True)
    
    # Calculate features for the last row
    df['returns'] = df['close'].pct_change()
    df['sma_20'] = df['close'].rolling(20).mean()
    df['sma_50'] = df['close'].rolling(50).mean()
    df['volatility'] = df['returns'].rolling(20).std() * np.sqrt(252)
    df['close_vs_sma20'] = (df['close'] - df['sma_20']) / df['sma_20']
    df['close_vs_sma50'] = (df['close'] - df['sma_50']) / df['sma_50']
    df['volume_ratio'] = df['volume'] / df['volume'].rolling(20).mean()
    
    df = df.dropna()
    last_row = df.iloc[-1]
    last_date = df.index[-1]
    current_price = last_row['close']
    
    # Create feature vector
    features = np.array([[
        last_row['open'], last_row['high'], last_row['low'], last_row['close'],
        last_row['volume'], last_row['returns'], last_row['volatility'],
        last_row['close_vs_sma20'], last_row['close_vs_sma50'], last_row['volume_ratio']
    ]])
    
    # Preprocess
    features = imputer.transform(features)
    features = scaler.transform(features)
    
    # Predict
    prediction = model.predict(features)[0]
    
    # Determine recommendation and confidence
    if prediction > 0.02:
        confidence = "High"
        recommendation = "Strong BUY"
        signal = "STRONG BULLISH 📈📈"
    elif prediction > 0:
        confidence = "Medium"
        recommendation = "Cautious BUY"
        signal = "BULLISH 📈"
    elif prediction > -0.02:
        confidence = "Low"
        recommendation = "HOLD"
        signal = "NEUTRAL ➡️"
    else:
        confidence = "Medium"
        recommendation = "SELL"
        signal = "BEARISH 📉"
    
    print(f"\n📊 Market Data (as of {last_date.strftime('%Y-%m-%d')}):")
    print(f"   Current S&P 500: ${current_price:,.2f}")
    
    print(f"\n🎯 PREDICTION:")
    print(f"   Expected Next Week Return: {prediction:.4%}")
    print(f"   Signal: {signal}")
    print(f"   Confidence: {confidence}")
    print(f"   Recommendation: {recommendation}")
    
    return {
        'prediction': prediction,
        'prediction_percent': f"{prediction:.4%}",
        'direction': 'BULLISH' if prediction > 0 else 'BEARISH',
        'signal': signal,
        'confidence': confidence,
        'recommendation': recommendation,
        'current_price': float(current_price),
        'date': last_date.strftime('%Y-%m-%d')
    }


# Cell 8: Main Execution
print("\n" + "="*60)
print("🚀 RUNNING S&P 500 PREDICTOR PIPELINE")
print("="*60)

# Initialize orchestrator
orchestrator = PipelineOrchestrator()

# Check if model exists and load it safely
model_path = MODELS_PATH / 'production_model.pkl'

if model_path.exists():
    print("\n✅ Existing model found.")
    try:
        artifacts = orchestrator.trainer.load_model()
        print("✅ Model loaded successfully!")
    except Exception as e:
        print(f"⚠️ Could not load existing model: {e}")
        print("   Training new model...")
        train_from_local_data()
else:
    print("\n⚠️ No model found. Training new model from local data...")
    train_from_local_data()
    
# Cell 9: Make Final Prediction
print("\n" + "="*60)
print("🎯 FINAL PREDICTION")
print("="*60)

# Try using orchestrator first
try:
    prediction = orchestrator.run_prediction()
except Exception as e:
    print(f"⚠️ Orchestrator prediction failed: {e}")
    print("   Using fallback prediction function...")
    prediction = predict_from_local()

# Display final result
print("\n" + "="*60)
print("📊 FINAL SUMMARY")
print("="*60)
print(f"📈 Expected Next Week Return: {prediction.get('prediction_percent', 'N/A')}")
print(f"🎯 Direction: {prediction.get('direction', 'N/A')}")
print(f"💡 Recommendation: {prediction.get('recommendation', 'N/A')}")
print(f"📊 Confidence: {prediction.get('confidence', 'N/A')}")
print(f"💰 Current Price: ${prediction.get('current_price', 0):,.2f}")
print("="*60)


# Cell 10: Model Status Check (Fixed)
print("\n📊 MODEL STATUS")
print("="*40)

model_path = MODELS_PATH / 'production_model.pkl'

if model_path.exists():
    artifacts = joblib.load(model_path)
    print(f"✅ Model exists")
    
    # Safe key access
    training_date = artifacts.get('training_date', 'Unknown')
    print(f"   Training date: {training_date}")
    
    metrics = artifacts.get('metrics', {})
    rmse = metrics.get('test_rmse', 0)
    direction_acc = metrics.get('direction_accuracy', 0)
    
    print(f"   RMSE: {rmse:.4f} ({rmse*100:.2f}%)")
    print(f"   Direction Accuracy: {direction_acc:.2%}")
    print(f"   Features: {len(artifacts.get('feature_names', []))}")
else:
    print(f"❌ No model found at {model_path}")

print("\n" + "="*60)
print("✅ PIPELINE EXECUTION COMPLETE!")
print("="*60)

📁 Project root: C:\Users\nyvra\Downloads\sp500-predictor
📁 Data path: C:\Users\nyvra\Downloads\sp500-predictor\data
📁 Models path: C:\Users\nyvra\Downloads\sp500-predictor\models
📁 Processed data exists: True
✅ DataCollector ready
✅ TrainingPipeline ready
✅ PredictionPipeline ready
✅ PipelineOrchestrator ready

🚀 RUNNING S&P 500 PREDICTOR PIPELINE

✅ Existing model found.
✅ Model loaded (RMSE: 0.0000)
✅ Model loaded successfully!

🎯 FINAL PREDICTION

🔮 Making prediction from local data...
✅ Model loaded (RMSE: 0.0000)

📊 Market Data (as of 2026-04-09):
   Current S&P 500: $679.91

🎯 PREDICTION:
   Expected Next Week Return: 0.2594%
   Direction: BULLISH
   Confidence: Medium
   Recommendation: Cautious BUY

📊 FINAL SUMMARY
📈 Expected Next Week Return: 0.2594%
🎯 Direction: BULLISH
💡 Recommendation: Cautious BUY
📊 Confidence: Medium
💰 Current Price: $679.91

📊 MODEL STATUS
✅ Model exists
   Training date: Unknown
   RMSE: 0.0000 (0.00%)
   Direction Accuracy: 0.00%
   Features: 10

✅ PIP

# Imports and Setup

In [19]:
import sys
import os
import json
import yaml
import pickle
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional, Any
import warnings
warnings.filterwarnings('ignore')

# ML
import yfinance as yf
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

# Models
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Drift detection
from scipy import stats
from scipy.spatial.distance import jensenshannon

# Set paths
PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data'
MODELS_PATH = PROJECT_ROOT / 'models'
PIPELINE_PATH = PROJECT_ROOT / 'pipeline'
LOGS_PATH = PROJECT_ROOT / 'logs'
DRIFT_PATH = PROJECT_ROOT / 'monitoring'

# Create directories
PIPELINE_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)
DRIFT_PATH.mkdir(parents=True, exist_ok=True)

print("✅ Pipeline imports loaded")
print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📁 Pipeline path: {PIPELINE_PATH}")
print(f"📁 Logs path: {LOGS_PATH}")
print(f"📁 Drift path: {DRIFT_PATH}")

✅ Pipeline imports loaded
📁 Project root: C:\Users\nyvra\Downloads\sp500-predictor
📁 Pipeline path: C:\Users\nyvra\Downloads\sp500-predictor\pipeline
📁 Logs path: C:\Users\nyvra\Downloads\sp500-predictor\logs
📁 Drift path: C:\Users\nyvra\Downloads\sp500-predictor\monitoring


# Data Collection Module

In [20]:
class DataCollector:
    """Collect and prepare data for training"""
    
    def __init__(self):
        self.imputer = SimpleImputer(strategy='mean')
        self.scaler = StandardScaler()
        
    def fetch_data(self, start_date: str = '2010-01-01', end_date: str = None) -> pd.DataFrame:
        """Fetch S&P 500 data from Yahoo Finance"""
        if end_date is None:
            end_date = datetime.now().strftime('%Y-%m-%d')
        
        print(f"📊 Fetching data from {start_date} to {end_date}")
        
        ticker = yf.Ticker("^GSPC")
        df = ticker.history(start=start_date, end=end_date)
        
        if df.empty:
            raise ValueError("No data fetched")
        
        df.columns = [col.lower() for col in df.columns]
        df['returns'] = df['close'].pct_change()
        df['target'] = df['close'].shift(-5) / df['close'] - 1  # Next week return
        
        df = df.dropna()
        
        print(f"✅ Fetched {len(df)} rows")
        return df
    
    def create_features(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, List[str]]:
        """Create feature matrix and target"""
        
        # Feature columns
        feature_cols = ['open', 'high', 'low', 'close', 'volume', 'returns']
        
        # Add technical indicators
        df['sma_20'] = df['close'].rolling(20).mean()
        df['sma_50'] = df['close'].rolling(50).mean()
        df['volatility'] = df['returns'].rolling(20).std() * np.sqrt(252)
        
        # Add price ratios
        df['close_vs_sma20'] = (df['close'] - df['sma_20']) / df['sma_20']
        df['close_vs_sma50'] = (df['close'] - df['sma_50']) / df['sma_50']
        
        # Add volume features
        df['volume_ratio'] = df['volume'] / df['volume'].rolling(20).mean()
        
        # Drop NaN from rolling calculations
        df = df.dropna()
        
        # Select final features
        final_features = ['open', 'high', 'low', 'close', 'volume', 'returns',
                         'volatility', 'close_vs_sma20', 'close_vs_sma50', 'volume_ratio']
        
        X = df[final_features].values
        y = df['target'].values
        
        print(f"✅ Created {len(final_features)} features, {len(y)} samples")
        
        return X, y, final_features
    
    def get_train_test_split(self, X: np.ndarray, y: np.ndarray, test_size: float = 0.2) -> Tuple:
        """Time-based train/test split"""
        split_idx = int(len(X) * (1 - test_size))
        
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]
        
        print(f"📊 Train: {len(X_train)} samples, Test: {len(X_test)} samples")
        
        return X_train, X_test, y_train, y_test

print("✅ DataCollector ready")

✅ DataCollector ready


# Model Training Pipeline

In [21]:
class TrainingPipeline:
    """End-to-end training pipeline"""
    
    def __init__(self):
        self.data_collector = DataCollector()
        self.model = None
        self.feature_names = None
        self.imputer = SimpleImputer(strategy='mean')
        self.scaler = StandardScaler()
        
    def train(self, start_date: str = '2010-01-01', retrain_reason: str = 'scheduled') -> Dict:
        """Execute full training pipeline"""
        
        print("\n" + "="*60)
        print("🚀 STARTING TRAINING PIPELINE")
        print(f"   Reason: {retrain_reason}")
        print("="*60)
        
        # Step 1: Collect data
        print("\n📊 Step 1: Collecting data...")
        df = self.data_collector.fetch_data(start_date)
        
        # Step 2: Create features
        print("\n🔧 Step 2: Creating features...")
        X, y, feature_names = self.data_collector.create_features(df)
        self.feature_names = feature_names
        
        # Step 3: Train/test split
        print("\n📊 Step 3: Splitting data...")
        X_train, X_test, y_train, y_test = self.data_collector.get_train_test_split(X, y)
        
        # Step 4: Preprocess
        print("\n🔧 Step 4: Preprocessing...")
        X_train = self.imputer.fit_transform(X_train)
        X_test = self.imputer.transform(X_test)
        X_train = self.scaler.fit_transform(X_train)
        X_test = self.scaler.transform(X_test)
        
        # Step 5: Train model
        print("\n🤖 Step 5: Training CatBoost model...")
        self.model = CatBoostRegressor(
            iterations=500,
            depth=6,
            learning_rate=0.1,
            random_seed=42,
            verbose=False
        )
        self.model.fit(X_train, y_train)
        
        # Step 6: Evaluate
        print("\n📈 Step 6: Evaluating model...")
        y_pred_train = self.model.predict(X_train)
        y_pred_test = self.model.predict(X_test)
        
        train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
        test_r2 = r2_score(y_test, y_pred_test)
        direction_acc = (np.sign(y_test) == np.sign(y_pred_test)).mean()
        
        print(f"   Train RMSE: {train_rmse:.4f}")
        print(f"   Test RMSE: {test_rmse:.4f}")
        print(f"   Test R²: {test_r2:.4f}")
        print(f"   Direction Accuracy: {direction_acc:.2%}")
        
        # Step 7: Save model and artifacts
        print("\n💾 Step 7: Saving artifacts...")
        artifacts = {
            'model': self.model,
            'imputer': self.imputer,
            'scaler': self.scaler,
            'feature_names': self.feature_names,
            'metrics': {
                'train_rmse': train_rmse,
                'test_rmse': test_rmse,
                'test_r2': test_r2,
                'direction_accuracy': direction_acc
            },
            'training_date': datetime.now().isoformat(),
            'retrain_reason': retrain_reason,
            'n_samples': len(y)
        }
        
        # Save using joblib
        model_path = MODELS_PATH / 'production_model.pkl'
        joblib.dump(artifacts, model_path)
        print(f"✅ Model saved to {model_path}")
        
        # Save metrics JSON
        metrics_path = MODELS_PATH / 'training_metrics.json'
        with open(metrics_path, 'w') as f:
            json.dump(artifacts['metrics'], f, indent=2)
        
        print("\n" + "="*60)
        print("✅ TRAINING PIPELINE COMPLETE")
        print("="*60)
        
        return artifacts
    
    def load_model(self) -> Dict:
        """Load production model"""
        model_path = MODELS_PATH / 'production_model.pkl'
        if not model_path.exists():
            raise FileNotFoundError(f"No production model found at {model_path}")
        
        artifacts = joblib.load(model_path)
        self.model = artifacts['model']
        self.imputer = artifacts['imputer']
        self.scaler = artifacts['scaler']
        self.feature_names = artifacts['feature_names']
        
        print(f"✅ Loaded model trained on {artifacts['training_date']}")
        return artifacts

print("✅ TrainingPipeline ready")

✅ TrainingPipeline ready


# Prediction Pipeline

In [22]:
class PredictionPipeline:
    """Real-time inference pipeline"""
    
    def __init__(self):
        self.model = None
        self.imputer = None
        self.scaler = None
        self.feature_names = None
        
    def load_model(self):
        """Load production model"""
        model_path = MODELS_PATH / 'production_model.pkl'
        if not model_path.exists():
            raise FileNotFoundError("No production model found. Run training first.")
        
        artifacts = joblib.load(model_path)
        self.model = artifacts['model']
        self.imputer = artifacts['imputer']
        self.scaler = artifacts['scaler']
        self.feature_names = artifacts['feature_names']
        print(f"✅ Model loaded (trained: {artifacts['training_date']})")
        
        return artifacts
    
    def get_latest_features(self) -> np.ndarray:
        """Get latest market features for prediction"""
        print("📊 Fetching latest market data...")
        
        # Fetch last 60 days for feature calculation
        ticker = yf.Ticker("^GSPC")
        df = ticker.history(period="60d")
        
        if df.empty:
            raise ValueError("Could not fetch market data")
        
        df.columns = [col.lower() for col in df.columns]
        df['returns'] = df['close'].pct_change()
        
        # Calculate features
        df['sma_20'] = df['close'].rolling(20).mean()
        df['sma_50'] = df['close'].rolling(50).mean()
        df['volatility'] = df['returns'].rolling(20).std() * np.sqrt(252)
        df['close_vs_sma20'] = (df['close'] - df['sma_20']) / df['sma_20']
        df['close_vs_sma50'] = (df['close'] - df['sma_50']) / df['sma_50']
        df['volume_ratio'] = df['volume'] / df['volume'].rolling(20).mean()
        
        # Get latest values
        latest = df.iloc[-1]
        features = np.array([
            latest['open'], latest['high'], latest['low'], latest['close'],
            latest['volume'], latest['returns'], latest['volatility'],
            latest['close_vs_sma20'], latest['close_vs_sma50'], latest['volume_ratio']
        ]).reshape(1, -1)
        
        print(f"✅ Features extracted for {df.index[-1].strftime('%Y-%m-%d')}")
        return features
    
    def predict(self, features: np.ndarray = None) -> Dict:
        """Make prediction"""
        if self.model is None:
            self.load_model()
        
        if features is None:
            features = self.get_latest_features()
        
        # Preprocess
        features = self.imputer.transform(features)
        features = self.scaler.transform(features)
        
        # Predict
        prediction = self.model.predict(features)[0]
        
        # Calculate confidence (based on model's prediction variance)
        confidence = min(95, max(50, 100 - abs(prediction) * 1000))
        
        result = {
            'prediction': prediction,
            'direction': 'BULLISH' if prediction > 0 else 'BEARISH',
            'confidence': confidence,
            'timestamp': datetime.now().isoformat(),
            'features': features[0].tolist() if features is not None else None
        }
        
        # Save prediction to log
        log_path = LOGS_PATH / 'predictions.log'
        with open(log_path, 'a') as f:
            f.write(json.dumps(result) + '\n')
        
        return result
    
    def batch_predict(self, X: np.ndarray) -> np.ndarray:
        """Make batch predictions"""
        if self.model is None:
            self.load_model()
        
        X_processed = self.imputer.transform(X)
        X_processed = self.scaler.transform(X_processed)
        return self.model.predict(X_processed)

print("✅ PredictionPipeline ready")

✅ PredictionPipeline ready


# Drift Monitoring Module

In [23]:
class DriftMonitor:
    """Monitor model drift using PSI and feature drift detection"""
    
    def __init__(self):
        self.baseline_distribution = None
        self.baseline_date = None
        
    def calculate_psi(self, expected: np.ndarray, actual: np.ndarray, bins: int = 10) -> float:
        """Calculate Population Stability Index"""
        # Bin the data
        expected_percents, bins_edges = np.histogram(expected, bins=bins, density=True)
        actual_percents, _ = np.histogram(actual, bins=bins_edges, density=True)
        
        # Add small constant to avoid division by zero
        expected_percents = np.clip(expected_percents, 1e-10, 1)
        actual_percents = np.clip(actual_percents, 1e-10, 1)
        
        # Calculate PSI
        psi = np.sum((actual_percents - expected_percents) * np.log(actual_percents / expected_percents))
        
        return psi
    
    def calculate_js_divergence(self, p: np.ndarray, q: np.ndarray) -> float:
        """Calculate Jensen-Shannon divergence"""
        return jensenshannon(p, q)
    
    def set_baseline(self, X: np.ndarray):
        """Set baseline distribution from training data"""
        self.baseline_distribution = X.copy()
        self.baseline_date = datetime.now()
        print(f"✅ Baseline set with {X.shape[0]} samples")
    
    def detect_drift(self, X_current: np.ndarray, threshold: float = 0.1) -> Dict:
        """Detect drift in current data"""
        if self.baseline_distribution is None:
            raise ValueError("No baseline set. Call set_baseline first.")
        
        results = {
            'drift_detected': False,
            'feature_drifts': {},
            'overall_psi': 0,
            'timestamp': datetime.now().isoformat()
        }
        
        # Calculate PSI for each feature
        psi_values = []
        for i in range(self.baseline_distribution.shape[1]):
            psi = self.calculate_psi(
                self.baseline_distribution[:, i],
                X_current[:, i]
            )
            psi_values.append(psi)
            
            if psi > threshold:
                results['feature_drifts'][f'feature_{i}'] = {
                    'psi': psi,
                    'severity': 'high' if psi > 0.25 else 'medium' if psi > 0.1 else 'low'
                }
                results['drift_detected'] = True
        
        results['overall_psi'] = np.mean(psi_values)
        
        # Determine drift severity
        if results['overall_psi'] > 0.25:
            results['severity'] = 'high'
        elif results['overall_psi'] > 0.1:
            results['severity'] = 'medium'
        else:
            results['severity'] = 'low'
        
        return results
    
    def monitor_predictions(self, predictions: np.ndarray, actuals: np.ndarray = None) -> Dict:
        """Monitor prediction drift"""
        results = {
            'timestamp': datetime.now().isoformat(),
            'prediction_stats': {
                'mean': float(np.mean(predictions)),
                'std': float(np.std(predictions)),
                'min': float(np.min(predictions)),
                'max': float(np.max(predictions))
            }
        }
        
        if actuals is not None:
            results['accuracy'] = {
                'rmse': float(np.sqrt(mean_squared_error(actuals, predictions))),
                'mae': float(mean_absolute_error(actuals, predictions)),
                'direction_accuracy': float((np.sign(actuals) == np.sign(predictions)).mean())
            }
        
        # Save monitoring results
        monitor_path = DRIFT_PATH / f'monitor_{datetime.now().strftime("%Y%m%d")}.json'
        with open(monitor_path, 'w') as f:
            json.dump(results, f, indent=2)
        
        return results

print("✅ DriftMonitor ready")

✅ DriftMonitor ready


# Complete Pipeline Orchestration

In [24]:

class PipelineOrchestrator:
    """Orchestrate full ML pipeline"""
    
    def __init__(self):
        self.trainer = TrainingPipeline()
        self.predictor = PredictionPipeline()
        self.drift_monitor = DriftMonitor()
        
    def run_full_training(self, retrain_reason: str = 'manual') -> Dict:
        """Run complete training pipeline"""
        print("\n🏗️ RUNNING FULL TRAINING PIPELINE")
        artifacts = self.trainer.train(retrain_reason=retrain_reason)
        
        # Set baseline for drift monitoring
        # Need to get training data for baseline
        df = self.trainer.data_collector.fetch_data()
        X, y, _ = self.trainer.data_collector.create_features(df)
        X = self.trainer.imputer.fit_transform(X)
        X = self.trainer.scaler.fit_transform(X)
        self.drift_monitor.set_baseline(X)
        
        return artifacts
    
    def run_prediction(self) -> Dict:
        """Run prediction pipeline"""
        print("\n🔮 RUNNING PREDICTION PIPELINE")
        result = self.predictor.predict()
        
        print(f"\n📊 Prediction Result:")
        print(f"   Next week return: {result['prediction']:.4%}")
        print(f"   Direction: {result['direction']}")
        print(f"   Confidence: {result['confidence']:.1f}%")
        
        return result
    
    def run_drift_check(self, X_current: np.ndarray) -> Dict:
        """Run drift detection"""
        print("\n📊 RUNNING DRIFT DETECTION")
        drift_result = self.drift_monitor.detect_drift(X_current)
        
        if drift_result['drift_detected']:
            print(f"⚠️ DRIFT DETECTED! Severity: {drift_result['severity']}")
            print(f"   Overall PSI: {drift_result['overall_psi']:.3f}")
        else:
            print(f"✅ No significant drift detected (PSI: {drift_result['overall_psi']:.3f})")
        
        return drift_result

# Initialize orchestrator
orchestrator = PipelineOrchestrator()
print("✅ PipelineOrchestrator ready")

# Quick test
print("\n🔍 Quick pipeline test:")
print("   - Run full training: orchestrator.run_full_training()")
print("   - Run prediction: orchestrator.run_prediction()")
print("   - Run drift check: orchestrator.run_drift_check(X_sample)")

✅ PipelineOrchestrator ready

🔍 Quick pipeline test:
   - Run full training: orchestrator.run_full_training()
   - Run prediction: orchestrator.run_prediction()
   - Run drift check: orchestrator.run_drift_check(X_sample)
